# Before switch-on, surface motion, and the evolving spot
## Emitter-driven three-fluid viscous diagnostic

Start here to see **both unforced interfaces**, the optical spot before switch-on, and the actual saved forward motion. Notebook 09 separately presents the synthesized stationary equilibrium and ultrasonic wavefronts.

The evolution uses the saved source amplitudes/phases with a declared 10 s ramp. At each accepted time step the acoustic field is recomputed on the current surfaces. The surfaces are never interpolated toward the target or replaced by the saved 0.513 µm equilibrium.

**Model:** axisymmetric three-fluid creeping flow, moving faceted fluid mesh, no-slip walls, nonlinear capillarity, gravity, fixed phase volumes and instantaneous harmonic acoustic forcing. Viscosities are the material-reference values (water / uncured NOA61 / water), not the earlier 5 Pa·s triplet. Inertia, convection, streaming and heating are omitted. Time and fluid-grid convergence have not been established, so this is a formation diagnostic, not a reliable settling-time prediction.

**Outcome:** this partial run did not form the desired lens. It was stopped at 7.1875 s after the conservative Reynolds estimate rose above the creeping-flow range. The maximum estimate reached 14.36; a phase-resolved inertial assessment is still needed. Saved frames beyond that warning remain visible as diagnostic data, not validated physical motion.


In [ ]:
%matplotlib inline
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from acoustic_freeform.dual.config import DualConfig
from acoustic_freeform.dual.surface import DualSurface, CartesianPatch
from acoustic_freeform.dual.optics import trace_pair
from acoustic_freeform.dual.dynamics import envelope

ROOT = next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').exists())
BASE = ROOT/'artifacts/noa61-emitter-2026-09-23'
RUN = BASE/'viscous-formation'
settings = json.loads((RUN/'config.json').read_text())
cfg = DualConfig(**settings['apparatus'])
material = settings['material']
space = DualSurface(cfg)
synthesis = json.loads((ROOT/settings['synthesis_config']).read_text())
targets = [CartesianPatch(cfg,j,**f) for j,f in enumerate(synthesis['case']['faces'])]
saved = np.load(RUN/'trajectory.npz')
times,states = saved['time_s'],saved['coefficients_m']
assert np.all(np.diff(times)>0) and np.max(abs(states[0])) == 0
driven = np.load(BASE/'verify-448-c4-clear-spot/design104/state.npz')['coefficients_m']
r = np.linspace(0,cfg.radius_m,801)
rp = np.linspace(0,cfg.clear_radius_m,401)
target_h = np.array([t.evaluate(r) for t in targets])
launch = cfg.levels_m[1]+targets[0].evaluate(rp)
traces = [trace_pair(space,q,material['indices'],material['stigmatic_z_m'][0],
    material['stigmatic_z_m'][2],launch_radius_m=rp,launch_height_m=launch) for q in states]
heights = np.array([[space.evaluate(v,r) for v in q] for q in states])
pupil = r <= cfg.clear_radius_m
errors = np.max(abs(heights[:,:,pupil]-target_h[:,pupil]),axis=2)
spots = np.array([t['max_radius_m'] for t in traces])
counts = np.array([t['transmitted'].sum() for t in traces])
colors = ['#1479b8','#e47d24']
plt.rcParams.update({'figure.dpi':110,'font.size':10})
print(f'Saved forward states: {len(times)}; physical time 0 to {times[-1]:.6g} s')
if (RUN/'validation.json').exists():
    status = json.loads((RUN/'validation.json').read_text())
    print('Scheduled run completed:',status['completed'],'; stop reason:',status['stop_reason'])
else:
    print('RUNNING SNAPSHOT: simulation is still running; this notebook includes only states already saved.')
print('Viscosities [Pa s]:',material['viscosities_pa_s'])
print('This is not a convergence-verified formation result.')


## 1. The equilibrium before emitters are on

For the declared volumes and pinned wall heights, **both unforced surfaces are flat**: lower z = −1 mm, upper z = +1 mm. Gravity does not imply a curved surface in this configuration: pressure is hydrostatic within each phase, and each flat interface lies at constant height. The constrained mechanical force at this state is zero.

The positive discrete stiffness check below establishes a local minimum within this axisymmetric surface space, not global or 3D stability. The dashed curves are the separately solved driven equilibrium—not the end of this trajectory unless the actual dynamics reaches it.


In [ ]:
initial_force,initial_stiffness,_ = space.mechanics(states[0])
assert np.max(abs(initial_force)) == 0
print('Unforced maximum generalized force [N]:',np.max(abs(initial_force)))
print('Smallest discrete stiffness eigenvalue [N/m]:',np.linalg.eigvalsh(initial_stiffness)[0])
fig,ax = plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
for j in range(2):
    ax[0].plot(r*1e3,(cfg.levels_m[j+1]+heights[0,j])*1e3,color=colors[j],
               label=['Front / lower: emitters OFF','Back / upper: emitters OFF'][j])
    ax[0].plot(r*1e3,(cfg.levels_m[j+1]+space.evaluate(driven[j],r))*1e3,'--',color=colors[j])
ax[0].set(xlabel='Radius [mm]',ylabel='z [mm]',title='Solid: unforced; dashed: driven equilibrium')
ax[0].legend(fontsize=8)
ax[1].scatter(*traces[0]['spots_m'].T*1e3,s=5,alpha=.6)
lim = max(spots[0]*1e3*1.1,.01)
ax[1].set(xlim=(-lim,lim),ylim=(-lim,lim),aspect='equal',
           xlabel='Detector x [mm]',ylabel='Detector y [mm]',
           title=f'BEFORE switch-on — {counts[0]}/{len(rp)} rays transmitted')
plt.show()


## 2. Actual forward evolution — embedded movie

Use Play or the slider below. Every frame corresponds to a **saved numerical state**; displayed time is physical simulation time. The source ramp changes the command, not the target shape.

Top left: both full surface profiles. Top middle: the current spot on a **fixed mm-scale detector window**. Top right: a fixed ±1 µm spot zoom. Bottom left: pupil height errors. Bottom middle: maximum spot radius and its current-time marker. Bottom right: commanded source envelope.

The small green circle is a 0.5 µm reference, not a pass badge. Lost rays remain counted. A blank µm zoom means the surviving rays are outside that field of view, not that the spot vanished. Both surface errors and optical radii refer to cycle-mean shapes. Playback uses equal frame intervals even when adaptive physical time steps differ; read the displayed physical time.


In [ ]:
fig,axes = plt.subplots(2,3,figsize=(15,8),constrained_layout=True)
profile,wide,zoom,errorax,historyax,commandax = axes.ravel()
lines=[]
for j in range(2):
    profile.plot(r*1e3,(cfg.levels_m[j+1]+target_h[j])*1e3,'--',color=colors[j],alpha=.6)
    lines.append(profile.plot([],[],color=colors[j],label=['Front / lower','Back / upper'][j])[0])
all_z = (heights + np.array(cfg.levels_m[1:3])[None,:,None])*1e3
profile.set(xlim=(0,4),ylim=(min(-1.4,float(all_z.min())-.1),max(1.4,float(all_z.max())+.1)),xlabel='Radius [mm]',ylabel='z [mm]',
            title='Solid: computed; dashed: target')
profile.legend(fontsize=8)
wide_points=wide.scatter([],[],s=5,alpha=.6)
wide_extent=max(float(np.nanmax(spots))*1e3*1.1,.01)
wide.set(xlim=(-wide_extent,wide_extent),ylim=(-wide_extent,wide_extent),aspect='equal',
         xlabel='Detector x [mm]',ylabel='Detector y [mm]')
zoom_points=zoom.scatter([],[],s=5)
zoom.add_patch(plt.Circle((0,0),.5,fill=False,color='green',ls='--'))
zoom.set(xlim=(-1,1),ylim=(-1,1),aspect='equal',xlabel='Detector x [µm]',
         ylabel='Detector y [µm]',title='Fixed ±1 µm zoom')
for j in range(2):
    errorax.semilogy(times,np.maximum(errors[:,j]*1e9,1e-5),color=colors[j])
errorax.axhline(10,color='green',ls='--')
errorax.set(xlabel='Physical time [s]',ylabel='Max sampled height error [nm]')
historyax.semilogy(times,np.maximum(spots*1e6,1e-6))
historyax.axhline(.5,color='green',ls='--')
historyax.set(xlabel='Physical time [s]',ylabel='Maximum surviving spot radius [µm]')
commandax.plot(times,[envelope(t,settings['ramp_s']) for t in times])
commandax.set(xlabel='Physical time [s]',ylabel='Source amplitude envelope',ylim=(-.03,1.03))
markers=[a.axvline(0,color='black',ls=':') for a in (errorax,historyax,commandax)]
heading=fig.suptitle('')
step_info=json.loads((RUN/'step-diagnostics.json').read_text())
reynolds=np.r_[0.,[d['reynolds_radius_upper_estimate'] for d in step_info]][:len(times)]
def update(k):
    for j in range(2):
        lines[j].set_data(r*1e3,(cfg.levels_m[j+1]+heights[k,j])*1e3)
    wide_points.set_offsets(traces[k]['spots_m']*1e3)
    zoom_points.set_offsets(traces[k]['spots_m']*1e6)
    wide.set_title(f'Current spot: {counts[k]}/{len(rp)} rays transmitted')
    for marker in markers: marker.set_xdata([times[k],times[k]])
    warning = ' — INERTIA WARNING' if reynolds[k] > 1 else ''
    heading.set_text(f'Physical time {times[k]:.4f} s — Stokes diagnostic{warning}\n'
                     f'Max spot {spots[k]*1e6:.3f} µm; front/back error '
                     f'{errors[k,0]*1e9:.1f}/{errors[k,1]*1e9:.1f} nm')
    return (*lines,wide_points,zoom_points,*markers,heading)
update(0)
animation=FuncAnimation(fig,update,frames=len(times),interval=180,blit=False)
embedded=animation.to_jshtml(embed_frames=True,default_mode='once')
plt.close(fig)
display(HTML(embedded))


## 3. Endpoint and validity checks

These curves describe what the computed trajectory did, whether or not it approaches the desired equilibrium. No stationary result is inserted as an extra animation frame.

The conservative Reynolds estimate uses the largest density/viscosity ratio across the three phases. The viscous diffusion time is based on the cylinder radius. Large values relative to the motion/ramp times warn against interpreting creeping-flow time as physical formation time.


In [ ]:
diagnostics=json.loads((RUN/'step-diagnostics.json').read_text())
if diagnostics:
    print('Maximum radius-based Reynolds upper estimate:',
          max(d['reynolds_radius_upper_estimate'] for d in diagnostics))
    print('Maximum mobility reciprocity error:',max(d['reciprocity_relative_error'] for d in diagnostics))
    print('Minimum frozen-load work minus energy change [J]:',
          min(d['work_minus_energy_change_j'] for d in diagnostics))
print('Largest phase viscous diffusion time [s]:',
      max(np.array(cfg.density_kg_m3)/material['viscosities_pa_s'])*cfg.radius_m**2)
print('Final sampled maximum height errors [nm]:',errors[-1]*1e9)
print('Final max surviving spot radius [µm]:',spots[-1]*1e6)
print('Final transmitted rays:',counts[-1],'/',len(rp))
print('No time-step, fluid-grid, stability or 10 nm certificate is implied.')


[Stationary emitter notebook and wavefronts](09_emitter_driven_equilibrium.ipynb) ·
[Run data](../artifacts/noa61-emitter-2026-09-23/viscous-formation/) ·
[Model and stepping protocol](../docs/emitter-formation-protocol.md)

The earlier prescribed-load notebook 08 is a different model and is not used to animate this emitter configuration.
